In [1]:
import os

In [2]:
os.chdir("../")

In [ ]:
import pandas as pd
from dataclasses import dataclass
from pathlib import Path
from sklearn.model_selection import train_test_split
from src.GNNClassfier.constants import *
from src.GNNClassfier.utils.common import read_yaml, create_directories
from src.GNNClassfier import logger

In [5]:
# 1. Configuration Entity
@dataclass
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    train_file: Path
    test_file: Path

In [6]:
# 2. Updated Configuration Manager
class ConfigurationManager:
    def __init__(self, config_filepath=CONFIG_FILE_PATH, params_filepath=PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation
        create_directories([config.root_dir])

        return DataTransformationConfig(
            root_dir=Path(config.root_dir),
            data_path=Path(config.data_path),
            train_file=Path(config.train_file),
            test_file=Path(config.test_file)
        )

In [7]:
# 3. Component Class
class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config

    def oversample_and_split(self):
        try:
            # Load dataset
            df = pd.read_csv(self.config.data_path)
            logger.info(f"Loaded dataset from {self.config.data_path}. Shape: {df.shape}")

            # Train-Test Split (before oversampling to prevent data leakage)
            train, test = train_test_split(df, test_size=0.2, random_state=42, stratify=df["HIV_active"])
            logger.info("Split data into train and test sets with stratification.")

            # Apply Oversampling to the TRAIN set only
            neg_class_count = train["HIV_active"].value_counts()[0]
            pos_class_count = train["HIV_active"].value_counts()[1]
            multiplier = int(neg_class_count / pos_class_count) - 1

            replicated_pos = [train[train["HIV_active"] == 1]] * multiplier
            train_oversampled = pd.concat([train] + replicated_pos, ignore_index=True)
            
            # Shuffle and reset index
            train_oversampled = train_oversampled.sample(frac=1, random_state=42).reset_index(drop=True)
            train_oversampled["index"] = train_oversampled.index
            
            logger.info(f"Oversampling complete. New train shape: {train_oversampled.shape}")

            # Save files
            train_oversampled.to_csv(self.config.train_file, index=False)
            test.to_csv(self.config.test_file, index=False)
            
            logger.info(f"Saved train and test files to {self.config.root_dir}")

        except Exception as e:
            raise e

In [9]:
# 4. Pipeline Execution
try:
    config_manager = ConfigurationManager()
    data_transformation_config = config_manager.get_data_transformation_config()
    data_transformation = DataTransformation(config=data_transformation_config)
    data_transformation.oversample_and_split()
except Exception as e:
    logger.exception(e)
    raise e

[2026-04-09 16:54:41,712: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-09 16:54:41,730: INFO: common: created directory at : artifacts]
[2026-04-09 16:54:41,734: INFO: common: created directory at : artifacts/data_transformation]
[2026-04-09 16:54:42,199: INFO: 2786068686: Loaded dataset from artifacts\data_ingestion\raw\hiv_dataset\HIV.csv. Shape: (41127, 3)]
[2026-04-09 16:54:42,347: INFO: 2786068686: Split data into train and test sets with stratification.]
[2026-04-09 16:54:42,558: INFO: 2786068686: Oversampling complete. New train shape: (62905, 4)]
[2026-04-09 16:54:42,945: INFO: 2786068686: Saved train and test files to artifacts\data_transformation]
